# Assignment 9

1) Environment
The environment is the 3‑pile Nim game with piles taking values 0–10. A state is [s0,s1,s2]; an episode ends at [0,0,0]. Actions remove 1–10 items from exactly one pile, yielding deterministic transitions and a fully observable, turn‑based, episodic MDP. Opponent moves (Random, Guru, or another learner) are part of the environment’s dynamics. Episodes start from random states.

2) Agents (is Guru an agent?)
There are three agents: Random, Guru, and the Q‑learner. Each maps the current state to an action. The Guru is indeed an agent—it follows a fixed, hand‑coded optimal Nim‑sum policy rather than a learned one.

3) Reward and penalty
The learner receives +100 only on the winning final move. All other moves receive 0, and there is no explicit penalty for the losing side. Q-values are updated with the specified learning rate and discount to back-propagate the reward through preceding states.

In our improved model the learner gets +100 for a win, and we penalize the losing side’s moves with −100; all intermediate moves otherwise get 0. Updates use learning rate α and discount γ to propagate terminal outcomes through the Q‑values.

4) Number of states
With 11 possible values per pile (0–10) and order mattering, there are 11³ = 1,331 distinct states.

5) Number of unique first actions
From [10,10,10], the player can choose one of 3 piles and remove 1–10 items, for 3×10 = 30 unique actions (matching the 30 actions per state in the Q‑table).

6) Can a Q‑learner beat the Guru?
A Q‑learner can, in theory, learn the same optimal play that the Guru uses. In Nim, if both players play perfectly, the side with a winning starting position will win. Our Q‑learner mostly trains against Random and only gets rewarded at the end of each game, so it learns to beat Random but not the Guru’s perfect play. If we mixed in training games against the Guru and update values after each move, it could get closer to that best strategy.

In [1]:
import numpy as np
from random import randint, choice

# The number of piles is 3


# max number of items per pile
ITEMS_MX = 10

# Initialize starting position
def init_game()->list:
    return [randint(1,ITEMS_MX), randint(1,ITEMS_MX), randint(1,ITEMS_MX)]

# Based on X-oring the item counts in piles - mathematical solution
def nim_guru(_st:list)->(int,int):
    xored = _st[0] ^ _st[1] ^ _st[2]
    if xored == 0:
        return nim_random(_st)
    for pile in range(3):
        s = _st[pile] ^ xored
        if s <= _st[pile]:
            return _st[pile]-s, pile

# Random Nim player
def nim_random(_st:list)->(int,int):
    pile = choice([i for i in range(3) if _st[i]>0])  # find the non-empty piles
    return randint(1, _st[pile]), pile  # random move

In [2]:
def given_nim_qlearner(_st:list)->(int,int):
    global qtable
    # pick the best rewarding move, equation 1
    a = np.argmax(qtable[_st[0], _st[1], _st[2]])  # exploitation
    # index is based on move, pile
    move, pile = a%ITEMS_MX+1, a//ITEMS_MX
    # check if qtable has generated a random but game illegal move - we have not explored there yet
    if move <= 0 or _st[pile] < move:
        move, pile = nim_random(_st)  # exploration
    return move, pile  # action

In [3]:
def nim_qlearner(_st:list)->(int,int):
    global qtable
    q_vals = qtable[_st[0], _st[1], _st[2]].copy()
    # mask illegal actions
    for a in range(ITEMS_MX*3):
        move, pile = a % ITEMS_MX + 1, a // ITEMS_MX
        if move > _st[pile]:
            q_vals[a] = -1e9
    a = int(np.argmax(q_vals))
    # if nothing learned yet for legal actions, explore
    if q_vals[a] <= 0:
        return nim_random(_st)
    move, pile = a % ITEMS_MX + 1, a // ITEMS_MX
    return move, pile

In [4]:
Engines = {'Random':nim_random, 'Guru':nim_guru, 'Qlearner':nim_qlearner}

def game(_a:str, _b:str):
    state, side = init_game(), 'A'
    while True:
        engine = Engines[_a] if side == 'A' else Engines[_b]
        move, pile = engine(state)
        # print(state, move, pile)  # debug purposes
        state[pile] -= move
        if state == [0, 0, 0]:  # game ends
            return side  # winning side
        side = 'B' if side == 'A' else 'A'  # switch sides

def play_games(_n:int, _a:str, _b:str)->(int,int):
    from collections import defaultdict
    wins = defaultdict(int)
    for _ in range(_n):
        wins[game(_a, _b)] += 1
    # info
    print(f"{_n} games, {_a:>8s}{wins['A']:5d}  {_b:>8s}{wins['B']:5d}")
    return wins['A'], wins['B']

In [5]:
# Play games
play_games(1000, 'Random', 'Random')
play_games(1000, 'Guru', 'Random')
play_games(1000, 'Random', 'Guru')
play_games(1000, 'Guru', 'Guru') ;

1000 games,   Random  478    Random  522
1000 games,     Guru  997    Random    3
1000 games,   Random    9      Guru  991
1000 games,     Guru  948      Guru   52


In [6]:
qtable, Alpha, Gamma, Reward = None, 1.0, 0.8, 100.0

# learn from _n games, randomly played to explore the possible states
def given_nim_qlearn(_n:int):
    global qtable
    # based on max items per pile
    qtable = np.zeros((ITEMS_MX+1, ITEMS_MX+1, ITEMS_MX+1, ITEMS_MX*3), dtype=np.float32)
    # play _n games
    for _ in range(_n):
        # first state is starting position
        st1 = init_game()
        while True:  # while game not finished
            # make a random move - exploration
            move, pile = nim_random(st1)
            st2 = list(st1)
            # make the move
            st2[pile] -= move  # --> last move I made
            if st2 == [0, 0, 0]:  # game ends
                given_qtable_update(Reward, st1, move, pile, 0)  # I won
                break  # new game

            given_qtable_update(0, st1, move, pile, np.max(qtable[st2[0], st2[1], st2[2]]))
            
            # Switch sides for play and learning
            st1 = st2
# Equation 3 - update the qtable
def given_qtable_update(r:float, _st1:list, move:int, pile:int, q_future_best:float):
    a = pile*ITEMS_MX+move-1
    qtable[_st1[0], _st1[1], _st1[2], a] = Alpha * (r + Gamma * q_future_best)


In [7]:
qtable, Alpha, Gamma, Reward = None, 0.17, 0.8, 100.0

def nim_qlearn(_n:int):
    global qtable
    # based on max items per pile
    qtable = np.zeros((ITEMS_MX+1, ITEMS_MX+1, ITEMS_MX+1, ITEMS_MX*3), dtype=np.float32)
    # play _n games
    for _ in range(_n):
        # Track the game history: list of (state, move, pile) tuples
        history = []
        st1 = init_game()
        
        while True:  # while game not finished
            # make a random move - exploration
            move, pile = nim_random(st1)
            
            # Store this move in history
            history.append((list(st1), move, pile))
            
            st2 = list(st1)
            # make the move
            st2[pile] -= move
            
            if st2 == [0, 0, 0]:  # game ends
                # Current player wins - update their moves with positive reward
                for idx, (hist_st, hist_move, hist_pile) in enumerate(history):
                    if idx % 2 == (len(history) - 1) % 2:  # same player as winner
                        qtable_update(Reward, hist_st, hist_move, hist_pile, 0)
                    else:  # losing player
                        qtable_update(-100, hist_st, hist_move, hist_pile, 0)  # NEGATIVE REWARD
                break
            
            # Continue to next state
            st1 = st2

# Equation 3 - update the qtable
def qtable_update(r:float, _st1:list, move:int, pile:int, q_future_best:float):
    a = pile*ITEMS_MX+move-1
    
    # Get the current Q-value for this state-action pair
    current_q = qtable[_st1[0], _st1[1], _st1[2], a]
    
    # Apply the Bellman equation: Q(s,a) = Q(s,a) + α[r + γ*max(Q(s',a')) - Q(s,a)]
    qtable[_st1[0], _st1[1], _st1[2], a] = current_q + Alpha * (r + Gamma * q_future_best - current_q)

In [8]:
nim_qlearn(1000)

In [9]:
	
# Play games
play_games(1000, 'Qlearner', 'Random')
play_games(1000, 'Random', 'Qlearner')

play_games(1000, 'Random', 'Random') ;

1000 games, Qlearner  857    Random  143
1000 games,   Random  141  Qlearner  859
1000 games,   Random  526    Random  474


In [10]:
%%time

# See the training size effect
n_train = (3, 10, 100, 1000, 10000, 50000, 100000)
Wins = []
for n in n_train:
    given_nim_qlearn(n)
    wins_a, wins_b = play_games(1000, 'Qlearner', 'Random')
    Wins += [wins_a/(wins_a+wins_b)]

print("\nGiven Q-learning:")
print(Wins)

1000 games, Qlearner  539    Random  461
1000 games, Qlearner  446    Random  554
1000 games, Qlearner  719    Random  281
1000 games, Qlearner  727    Random  273
1000 games, Qlearner  713    Random  287
1000 games, Qlearner  691    Random  309
1000 games, Qlearner  717    Random  283

Given Q-learning:
[0.539, 0.446, 0.719, 0.727, 0.713, 0.691, 0.717]
CPU times: user 1.68 s, sys: 36 ms, total: 1.71 s
Wall time: 1.68 s


In [11]:
%%time
print("\nMy updated Q-learning:")

# See the training size effect
n_train = (3, 10, 100, 1000, 10000, 50000, 100000)
Wins = []
for n in n_train:
    nim_qlearn(n)
    wins_a, wins_b = play_games(1000, 'Qlearner', 'Random')
    Wins += [wins_a/(wins_a+wins_b)]

print(Wins)


My updated Q-learning:
1000 games, Qlearner  526    Random  474
1000 games, Qlearner  573    Random  427
1000 games, Qlearner  743    Random  257
1000 games, Qlearner  869    Random  131
1000 games, Qlearner  922    Random   78
1000 games, Qlearner  918    Random   82
1000 games, Qlearner  914    Random   86
[0.526, 0.573, 0.743, 0.869, 0.922, 0.918, 0.914]
CPU times: user 1.04 s, sys: 31.3 ms, total: 1.07 s
Wall time: 1.04 s


In [12]:
%%time
print("\nMy updated Q-learning:")

# See the training size effect
n_train = (3, 10, 100, 1000, 10000, 50000, 100000)
Wins = []
for n in n_train:
    nim_qlearn(n)
    wins_a, wins_b = play_games(1000, 'Qlearner', 'Guru')
    Wins += [wins_a/(wins_a+wins_b)]

print(Wins)


My updated Q-learning:
1000 games, Qlearner    9      Guru  991
1000 games, Qlearner    9      Guru  991
1000 games, Qlearner   18      Guru  982
1000 games, Qlearner   29      Guru  971
1000 games, Qlearner   60      Guru  940
1000 games, Qlearner   50      Guru  950
1000 games, Qlearner   46      Guru  954
[0.009, 0.009, 0.018, 0.029, 0.06, 0.05, 0.046]
CPU times: user 1.02 s, sys: 12.9 ms, total: 1.04 s
Wall time: 1.03 s


In [13]:

Wins = []
for n in n_train:
    given_nim_qlearn(n)
    wins_a, wins_b = play_games(1000, 'Qlearner', 'Guru')
    Wins += [wins_a/(wins_a+wins_b)]

print("\nMy updated Q-learning:")
print(Wins)

1000 games, Qlearner   10      Guru  990
1000 games, Qlearner    7      Guru  993
1000 games, Qlearner    5      Guru  995
1000 games, Qlearner    9      Guru  991
1000 games, Qlearner   32      Guru  968
1000 games, Qlearner   24      Guru  976
1000 games, Qlearner   30      Guru  970

My updated Q-learning:
[0.01, 0.007, 0.005, 0.009, 0.032, 0.024, 0.03]


I updated the Nim Q‑learner in four ways. First, I made action selection safe by masking illegal actions and choosing the best legal one; if nothing looks good yet, it falls back to a random legal move. Second, I added loss penalties during training by recording the move history and, at the end of each game, giving +100 to the winner’s moves and −100 to the loser’s moves. Third, I switched from overwriting Q‑values to an incremental update that nudges each value toward the outcome and expected value of the next state. Finally, I set the main hyperparameters to a learning rate of 0.17, a discount factor of 0.8, and a terminal reward of 100.